# Baseline models

This notebook evaluates the benchmark forecasting models on the
cleaned train, validation and test splits. Simple statistical models are also fit directly here.

All baseline models return predictions and ground truth in raw value space.
`ForecastEvaluator` is responsible for transforming predictions into the
required evaluation space and computing the common metrics.

The current available evaluation metrics are:

1. **Cumulative log-change MAE**
2. **MASE**
3. **Pearson Correlation between predictions and target in cumulative log change space**
4. **Relative MAE versus Persistence**
5. **Persistence win rate**

Bootstrap evaluation metrics over the test datset are available and used by default.

The available benchmark models are:

1. **Persistence** — predicts every future horizon using the final target
   value in the context window.
2. **Mean** — predicts every future horizon using the mean target value over
   the context window.
3. **ARIMA** — fits a separate univariate ARIMA model to the one-step log
   changes of each asset and target channel.
4. **VAR** — fits one multivariate VAR model per target channel across all
   assets.
5. **GARCH** — fits a separate GARCH(1,1) model to each asset and target
   channel, with an optional AR(1), constant or zero conditional mean.
6. **ModernTCN** - multiple ablations have been trained in Colab. The
    best version checkpoint (based on validation loss) is called and used to 
    predict. It is the version which takes all OHLCV as input, adds a time
    of day temporal feature and flattens series into batch (so we dont mix
    across asset - huge reduction in parameter count - 121,138 params in total
    including 256 params added for the temporal feature).


In [ ]:
from pathlib import Path
import sys
from time import perf_counter
import pandas as pd
import torch

# Make sure notebook can import from src/
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

from src.data.load_candle_data import load_candle_splits, clean_candle_splits
from src.evaluation.metrics import ForecastEvaluator
from src.models.persistence import PersistenceBaseline
from src.models.mean import MeanBaseline
from src.models.arima import ArimaBaseline
from src.models.var import VarBaseline
from src.models.garch import GarchBaseline
from src.models.modern_tcn import ModernTCNBaseline
from src.models.continuous_forecaster import (
    evaluate_saved_continuous_forecaster_run,
)
from src.utils.config import load_yaml
from src.utils.metric_tables import (
    DEFAULT_SUMMARY_METRICS,
    make_evaluation_table,
    make_baseline_summary_table,
)
from src.visualization.forecast_plots import plot_forecast_comparison


In [ ]:
DATA_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "Shared drives/Vishal/data/cached_datasets/"
    "exp-1m-95s-24y/session"
)

CONFIG_PATH = Path("../configs/forecasting.yaml")

BASELINE_CACHE_ROOT = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation"
)

## Load the data and clean

In [ ]:
config = load_yaml(CONFIG_PATH)

train_raw, val_raw, test_raw = load_candle_splits(DATA_DIR)

train, val, test = clean_candle_splits(
    train_raw,
    val_raw,
    test_raw,
)

print("train samples:", len(train["samples"]))
print("val samples:", len(val["samples"]))
print("test samples:", len(test["samples"]))
print("channels:", test["channels"])
print("assets:", len(test["asset_cols"]))
print("stride:", config['forecasting']['stride'])
print("input features:", config['forecasting']['input_channels'])
print("targets:", config['forecasting']['target_channels'])

#Set global Bootstrap params
BOOTSTRAP_N = 10_000
BOOTSTRAP_CONFIDENCE_LEVEL = 0.95
BOOTSTRAP_SEED = 42


## Summary metrics and forecast plots

Run the individual model sections first so that each `{model}_metric_table` exists, then rerun the summary cell below. The summary contains ordinary full-test-set values. Bootstrap confidence intervals remain available in each detailed model table.

In [25]:
models_to_display = [
    "persistence",
    "arima",
    "var",
    "garch",
    "modern_tcn",
    "kronos",
    "final_model",
]

baseline_summary_table = make_baseline_summary_table(
    models_to_display=models_to_display,
    namespace=globals(),
    channel="close",
    metrics_to_display=DEFAULT_SUMMARY_METRICS,
    model_display_names={
        "modern_tcn": "ModernTCN ",
        "final_model": "GeometricTCN",
    },
)

display(
    baseline_summary_table.style
    .format(
        "{:.6g}",
        na_rep="—",
    )
    .set_caption(
        "Frozen Test-Set Results"
    )
)

Use the function below to plot the forecasts of any model vs the true price. Can compare multiple models.

In [ ]:
fig, axes, selection = plot_forecast_comparison(
    models=["persistence","arima", "modern_tcn", "kronos", "final_model"],
    namespace=globals(),
    day=None,
    asset="NVDA",
    horizons=None,
)

print(selection)

## Persistence

In [ ]:
RUN = False

persistence_prediction_path = (
    BASELINE_CACHE_ROOT
    / "persistence"
    / "prediction_result.pt"
)

if RUN:
    persistence = PersistenceBaseline.from_config(
        config
    )

    persistence.fit(
        train_split=train,
        val_split=val,
    )

    persistence_result = persistence.predict(
        split=test,
        batch_size=256,
    )

    persistence_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        persistence_result,
        persistence_prediction_path,
    )

else:
    persistence_result = torch.load(
        persistence_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

persistence_evaluator = ForecastEvaluator(
    prediction_result=persistence_result,
    train_split=val,
)


# Persistence predicts zero cumulative log change at every horizon,
# so its cumulative-log-change Pearson correlation or IC is undefined.
persistence_undefined_metrics = {
    "cumulative_log_change_pearson_correlation",
    "cumulative_log_change_cross_sectional_pearson_ic",
    "cumulative_log_change_cross_sectional_spearman_rank_ic",
    "cumulative_log_change_temporal_absolute_correlation"

}

persistence_metric_names = [
    metric_name
    for metric_name in persistence_evaluator.available_metrics
    if metric_name not in persistence_undefined_metrics
]


persistence_results = persistence_evaluator.evaluate(
    metrics=persistence_metric_names,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

persistence_metric_table = make_evaluation_table(
    metric_results=persistence_results,
    horizons=persistence_evaluator.horizons,
    channels=persistence_evaluator.channels,
)


for metric_name in persistence_metric_names:
    metric_display = (
        persistence_metric_table
        .loc[
            persistence_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format("{:.6g}", na_rep="—")
        .set_caption(
            metric_name
        )
    )

## Mean

In [ ]:
RUN = False

mean_prediction_path = (
    BASELINE_CACHE_ROOT
    / "mean"
    / "prediction_result.pt"
)

if RUN:
    mean = MeanBaseline.from_config(
        config
    )

    mean.fit(
        train_split=train,
        val_split=val,
    )

    mean_result = mean.predict(
        split=test,
        batch_size=256,
    )

    mean_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        mean_result,
        mean_prediction_path,
    )

else:
    mean_result = torch.load(
        mean_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

mean_evaluator = ForecastEvaluator(
    prediction_result=mean_result,
    train_split=train,
)

mean_results = mean_evaluator.evaluate(
    metrics=mean_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

mean_metric_table = make_evaluation_table(
    metric_results=mean_results,
    horizons=mean_evaluator.horizons,
    channels=mean_evaluator.channels,
)

for metric_name in mean_evaluator.available_metrics:
    metric_display = (
        mean_metric_table
        .loc[
            mean_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format("{:.6g}", na_rep="—")
        .set_caption(
            metric_name
        )
    )

## ARIMA

In [ ]:
RUN = False

arima_prediction_path = (
    BASELINE_CACHE_ROOT
    / "arima"
    / "prediction_result.pt"
)

if RUN:
    arima = ArimaBaseline.from_config(
        config,
        fit_mode="simple",
        optim_method="powell",
    )

    arima.fit(
        train_split=train,
        val_split=val,
    )

    arima_result = arima.predict(
        split=test,
        batch_size=32,
    )

    arima_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        arima_result,
        arima_prediction_path,
    )

else:
    arima_result = torch.load(
        arima_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

arima_evaluator = ForecastEvaluator(
    prediction_result=arima_result,
    train_split=train,
)

arima_results = arima_evaluator.evaluate(
    metrics=arima_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

arima_metric_table = make_evaluation_table(
    metric_results=arima_results,
    horizons=arima_evaluator.horizons,
    channels=arima_evaluator.channels,
)

for metric_name in arima_evaluator.available_metrics:
    metric_display = (
        arima_metric_table
        .loc[
            arima_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format("{:.6g}", na_rep="—")
        .set_caption(
            metric_name
        )
    )

## VAR

In [ ]:
RUN = False

var_prediction_path = (
    BASELINE_CACHE_ROOT
    / "var"
    / "prediction_result.pt"
)

if RUN:
    var = VarBaseline.from_config(
        config,
        maxlags=15,
        ic="aic",
        trend="c",
    )

    var.fit(
        train_split=train,
        val_split=val,
    )

    var_result = var.predict(
        split=test,
        batch_size=256,
    )

    var_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        var_result,
        var_prediction_path,
    )

else:
    var_result = torch.load(
        var_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

var_evaluator = ForecastEvaluator(
    prediction_result=var_result,
    train_split=train,
)

var_results = var_evaluator.evaluate(
    metrics=var_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

var_metric_table = make_evaluation_table(
    metric_results=var_results,
    horizons=var_evaluator.horizons,
    channels=var_evaluator.channels,
)

for metric_name in var_evaluator.available_metrics:
    metric_display = (
        var_metric_table
        .loc[
            var_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format("{:.6g}", na_rep="—")
        .set_caption(
            metric_name
        )
    )

## GARCH

In [ ]:
RUN = False

garch_prediction_path = (
    BASELINE_CACHE_ROOT
    / "garch"
    / "prediction_result.pt"
)

if RUN:
    garch = GarchBaseline.from_config(
        config,
        mean="AR",
        return_scale=10000.0,
    )

    garch.fit(
        train_split=train,
        val_split=val,
    )

    garch_result = garch.predict(
        split=test,
        batch_size=256,
    )

    garch_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    torch.save(
        garch_result,
        garch_prediction_path,
    )

else:
    garch_result = torch.load(
        garch_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

garch_evaluator = ForecastEvaluator(
    prediction_result=garch_result,
    train_split=train,
)

garch_results = garch_evaluator.evaluate(
    metrics=garch_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

garch_metric_table = make_evaluation_table(
    metric_results=garch_results,
    horizons=garch_evaluator.horizons,
    channels=garch_evaluator.channels,
)

for metric_name in garch_evaluator.available_metrics:
    metric_display = (
        garch_metric_table
        .loc[
            garch_metric_table["metric"]
            == metric_name
        ]
        .set_index(
            [
                "horizon",
                "channel",
            ]
        )[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
    )

    display(
        metric_display.style
        .format("{:.6g}", na_rep="—")
        .set_caption(
            metric_name
        )
    )

## ModernTCN — joint OHLCV baseline trained with cumulative-log-change MAE

This cell reports the cross-asset ModernTCN baseline selected for the dissertation table. Set `MODERN_TCN_RUN_DIR` to any completed standalone ModernTCN run directory containing `best_checkpoint.pt`.

The checkpoint is now authoritative: its saved input channels, target channels, joint/per-asset layout, patch geometry, hidden dimension, kernels, RevIN/context-normalisation settings and temporal-position setting are restored automatically. The current `forecasting.yaml` may describe a different ModernTCN run without blocking inference.

In [26]:
# Inputs:
# - RUN=True: load the selected checkpoint, generate test predictions, and
#   overwrite BASELINE_CACHE_ROOT/modern_tcn/prediction_result.pt.
# - RUN=False: load that existing prediction file without running the model.
# - MODERN_TCN_RUN_DIR: any completed ModernTCN run folder containing
#   best_checkpoint.pt. No match to forecasting.yaml is required.

RUN = True

MODERN_TCN_RUN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/checkpoints/modern_tcn/"
    "modern_tcn_joint_ohlcv_clgmae_d32_k1_p4s2_lk51_fs1"
).expanduser().resolve()

modern_tcn_prediction_path = (
    BASELINE_CACHE_ROOT
    / "modern_tcn"
    / "prediction_result.pt"
)

if RUN:
    modern_tcn_checkpoint_path = (
        MODERN_TCN_RUN_DIR
        / "best_checkpoint.pt"
    )

    modern_tcn = ModernTCNBaseline.from_checkpoint(
        checkpoint_path=modern_tcn_checkpoint_path,
        device="cpu",
    )

    modern_tcn_result = modern_tcn.predict(
        split=test,
        batch_size=8,
        num_workers=0,
    )
    modern_tcn_result["asset_cols"] = list(test["asset_cols"])
    modern_tcn_result["output_space"] = "raw"

    modern_tcn_prediction_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )
    torch.save(
        modern_tcn_result,
        modern_tcn_prediction_path,
    )
else:
    modern_tcn_result = torch.load(
        modern_tcn_prediction_path,
        map_location="cpu",
        weights_only=False,
    )

modern_tcn_evaluator = ForecastEvaluator(
    prediction_result=modern_tcn_result,
    train_split=train,
)
modern_tcn_results = modern_tcn_evaluator.evaluate(
    metrics=modern_tcn_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)
modern_tcn_metric_table = make_evaluation_table(
    metric_results=modern_tcn_results,
    horizons=modern_tcn_evaluator.horizons,
    channels=modern_tcn_evaluator.channels,
)

for metric_name in modern_tcn_evaluator.available_metrics:
    display(
        modern_tcn_metric_table
        .loc[modern_tcn_metric_table["metric"] == metric_name]
        .set_index(["horizon", "channel"])[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style
        .format("{:.6g}", na_rep="—")
        .set_caption(metric_name)
    )

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000374363,0.000359787,0.000390145,0.0003744,7.78542e-06
5,close,0.000795578,0.000766865,0.000826125,0.0007956,1.52125e-05
15,close,0.00132698,0.00128055,0.00137625,0.00132709,2.45423e-05
30,close,0.00184512,0.00177227,0.00192292,0.00184544,3.86984e-05
60,close,0.00256613,0.00243826,0.00271272,0.00256684,7.05733e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000261545,—,—,—,—
5,close,0.000580192,—,—,—,—
15,close,0.000981331,—,—,—,—
30,close,0.00136471,—,—,—,—
60,close,0.00190544,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.00111389,—,—,—,—
5,close,0.0022378,—,—,—,—
15,close,0.00370741,—,—,—,—
30,close,0.00515032,—,—,—,—
60,close,0.00713348,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.013826,-0.00762673,0.0344435,0.0136802,0.0107975
5,close,0.0104933,-0.00676377,0.0279396,0.0104498,0.00880349
15,close,0.0142234,-0.00626076,0.0334795,0.0143467,0.0101159
30,close,0.0133318,-0.0104149,0.0382594,0.0135653,0.0124603
60,close,0.00656612,-0.0245052,0.035693,0.0071055,0.0155435


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.170854,0.166412,0.175059,0.17071,0.00221164
5,close,0.159404,0.156383,0.163099,0.15967,0.00172219
15,close,0.0985171,0.0962066,0.100289,0.0982399,0.00103743
30,close,0.100588,0.0983572,0.103565,0.100946,0.00132709
60,close,0.102837,0.0988803,0.10599,0.10273,0.00183258


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.970071,0.932327,1.01121,0.970174,0.0201585
5,close,2.06907,1.99427,2.14905,2.06915,0.0395734
15,close,3.44656,3.3261,3.57632,3.44689,0.063913
30,close,4.79536,4.60534,4.99911,4.7963,0.101211
60,close,6.68442,6.3536,7.06404,6.68645,0.181278


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.02519,1.0213,1.029,1.02519,0.00196434
5,close,1.01362,1.00945,1.01771,1.01362,0.00212512
15,close,1.00509,1.0027,1.00757,1.00509,0.00125172
30,close,1.00377,1.00099,1.00666,1.00379,0.0014547
60,close,1.00342,0.999953,1.00699,1.00341,0.00180373


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.442179,0.436342,0.448204,0.442185,0.00301442
5,close,0.472534,0.465684,0.479595,0.472517,0.00352491
15,close,0.483323,0.477668,0.489024,0.483333,0.00286023
30,close,0.485665,0.478791,0.492355,0.485638,0.00344751
60,close,0.482214,0.472972,0.490809,0.482167,0.00455999


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.01724,0.00744932,0.0265696,0.0172296,0.00489607
5,close,0.0116784,-0.000121495,0.0237308,0.0116531,0.00606724
15,close,0.0134228,0.00224994,0.0246288,0.0134125,0.00570986
30,close,0.0209041,0.00633276,0.0347555,0.0208112,0.00722879
60,close,0.0217686,0.00404158,0.0380542,0.0216536,0.00862931


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.022924,0.0161108,0.029942,0.0229358,0.00350777
5,close,0.0173566,0.0084832,0.0265048,0.0173304,0.00458958
15,close,0.0174739,0.00834711,0.0264183,0.017433,0.00457311
30,close,0.0235336,0.0115286,0.0348232,0.0234218,0.0059802
60,close,0.0193322,0.00457325,0.0331612,0.019195,0.00719166


## Final continuous dynamic-graph model

This cell evaluates a completed continuous-forecaster run directory on the held-out test split. The run folder is the source of truth: `resolved_config.json` reconstructs the exact temporal backbone, graph learner, spatial mixer, learned beta gate and output representation, while `best_checkpoint.pt` supplies the selected weights.

With `RUN=True`, the helper performs one chronological test inference pass and writes the following files beside `best_checkpoint.pt`:

- `test_predictions.pt`
- `test_graphs.pt`
- `test_metric_table.csv`
- `test_diagnostics.json`

With `RUN=False`, it reloads `test_predictions.pt` and recomputes the common metrics and bootstrap table without running the model again.

In [20]:
# Inputs:
# - RUN=True generates the test predictions once; RUN=False reloads them.
# - FINAL_MODEL_RUN_DIR is any completed run from run_continuous_forecaster.py.
# - FINAL_MODEL_DEVICE may be "cpu", "mps", "cuda", or "auto".
# - FINAL_MODEL_BATCH_SIZE controls inference memory only; it does not alter
#   model weights or predictions apart from normal floating-point variation.
RUN = True

FINAL_MODEL_RUN_DIR = Path(
    "/Users/vishalruparelia/Library/CloudStorage/"
    "GoogleDrive-vishal@autonomous-fox.ai/"
    "My Drive/dissertation/models_to_use/continuous_dynamic"
).expanduser().resolve()

FINAL_MODEL_DEVICE = "auto"
FINAL_MODEL_BATCH_SIZE = 32

final_model_evaluation = evaluate_saved_continuous_forecaster_run(
    run_dir=FINAL_MODEL_RUN_DIR,
    train_split=train,
    evaluation_split=test,
    split_name="test",
    run_inference=RUN,
    device=FINAL_MODEL_DEVICE,
    batch_size=FINAL_MODEL_BATCH_SIZE,
    num_workers=0,
    prediction_filename="test_predictions.pt",
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

final_model_result = final_model_evaluation.prediction_result
final_model_results = final_model_evaluation.metric_results
final_model_metric_table = final_model_evaluation.metric_table

print("Checkpoint epoch:", final_model_evaluation.checkpoint_epoch)
print("Predictions:", final_model_evaluation.prediction_path)
print("Graphs:", final_model_evaluation.graph_path)

for metric_name in final_model_results:
    display(
        final_model_metric_table
        .loc[final_model_metric_table["metric"] == metric_name]
        .set_index(["horizon", "channel"])[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style
        .format("{:.6g}", na_rep="—")
        .set_caption(metric_name)
    )

continuous_dynamic test selected-checkpoint inference:   0%|          | 0/37 [00:00<?, ?it/s]

Checkpoint epoch: 33
Predictions: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/models_to_use/continuous_dynamic/test_predictions.pt
Graphs: /Users/vishalruparelia/Library/CloudStorage/GoogleDrive-vishal@autonomous-fox.ai/My Drive/dissertation/models_to_use/continuous_dynamic/test_graphs.pt


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000369537,0.000354697,0.000385591,0.000369575,7.90252e-06
5,close,0.000787019,0.000758667,0.000817396,0.000787038,1.50501e-05
15,close,0.0013234,0.0012773,0.00137235,0.00132351,2.44264e-05
30,close,0.00184138,0.00176965,0.00191734,0.00184169,3.80145e-05
60,close,0.00255811,0.00243109,0.00270317,0.0025588,6.9924e-05


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.000257492,—,—,—,—
5,close,0.000575066,—,—,—,—
15,close,0.000979424,—,—,—,—
30,close,0.00136137,—,—,—,—
60,close,0.00190115,—,—,—,—


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0010972,—,—,—,—
5,close,0.00220871,—,—,—,—
15,close,0.0036947,—,—,—,—
30,close,0.00511154,—,—,—,—
60,close,0.00711966,—,—,—,—


...

,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.957835,0.919697,0.999788,0.957941,0.0203864
5,close,2.0471,1.97299,2.12628,2.04717,0.0390217
15,close,3.43689,3.31722,3.5654,3.43723,0.0635315
30,close,4.78425,4.5963,4.98569,4.78517,0.0996257
60,close,6.66267,6.33333,7.03835,6.66463,0.179346


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,1.0098,1.00801,1.01164,1.00982,0.000926183
5,close,1.00097,0.999201,1.00259,1.00096,0.00086552
15,close,1.00067,0.999264,1.00203,1.00066,0.000708467
30,close,1.00154,0.999861,1.00323,1.00155,0.000859473
60,close,1.00018,0.997829,1.00258,1.00019,0.00121617


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.451161,0.445324,0.45693,0.451177,0.00295193
5,close,0.48393,0.478339,0.489722,0.483935,0.00288417
15,close,0.489334,0.482785,0.496057,0.48936,0.00337345
30,close,0.487408,0.480201,0.494441,0.487405,0.0036359
60,close,0.48676,0.477057,0.496239,0.486727,0.00484654


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0351325,0.0252455,0.0445003,0.0351485,0.00494695
5,close,0.0220228,0.0115738,0.0326067,0.0220097,0.00535993
15,close,0.016547,0.00430923,0.0288486,0.0165109,0.00625127
30,close,0.0110757,-0.00231787,0.0244359,0.0109134,0.00687018
60,close,0.0179005,0.0033777,0.0316486,0.0176966,0.00722177


,,value,ci_lower,ci_upper,bootstrap_mean,bootstrap_std
horizon,channel,,,,,
1,close,0.0333385,0.0246297,0.0417424,0.0333514,0.00433921
5,close,0.019483,0.0114281,0.0278608,0.0194729,0.00420327
15,close,0.0125464,0.00374891,0.021566,0.0125325,0.0045686
30,close,0.0121105,0.00181891,0.0221674,0.0119896,0.00514409
60,close,0.0178917,0.00632608,0.0284351,0.0177297,0.00564536


## Kronos

In [ ]:
kronos_result = torch.load(
    Path(
        "/Users/vishalruparelia/Library/CloudStorage/"
        "GoogleDrive-vishal@autonomous-fox.ai/"
        "My Drive/dissertation/kronos/"
        "kronos_small_test_fp16_bs16_progress.pt"
    ),
    map_location="cpu",
    weights_only=False,
)["prediction_result"]

kronos_evaluator = ForecastEvaluator(
    prediction_result=kronos_result,
    train_split=train,
)

kronos_results = kronos_evaluator.evaluate(
    metrics=kronos_evaluator.available_metrics,
    reduce_dims=(0, 2),
    bootstrap=True,
    n_bootstrap=BOOTSTRAP_N,
    confidence_level=BOOTSTRAP_CONFIDENCE_LEVEL,
    bootstrap_seed=BOOTSTRAP_SEED,
)

kronos_metric_table = make_evaluation_table(
    metric_results=kronos_results,
    horizons=kronos_evaluator.horizons,
    channels=kronos_evaluator.channels,
)

for metric_name in kronos_evaluator.available_metrics:
    display(
        kronos_metric_table
        .loc[kronos_metric_table["metric"] == metric_name]
        .set_index(["horizon", "channel"])[
            [
                "value",
                "ci_lower",
                "ci_upper",
                "bootstrap_mean",
                "bootstrap_std",
            ]
        ]
        .style
        .format("{:.6g}", na_rep="—")
        .set_caption(metric_name)
    )